# Lecția 02 - Explorarea Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** este un cadru unificat pentru construirea de agenți AI. Oferă o arhitectură curată și compozabilă cu patru componente de bază:

- **Client** – se conectează la un endpoint al modelului AI și gestionează comunicarea
- **Agent** – învăluie un client cu instrucțiuni și definiții de unelte
- **Unelte** – extind capacitățile agentului cu funcții personalizate pe care modelul le poate apela
- **Sesiune** – păstrează istoricul conversațiilor pentru interacțiuni multiple

În această lecție, vom construi un **agent de rezervare călătorii** care verifică disponibilitatea destinațiilor folosind aceste concepte.


## Configurare


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Înțelegerea arhitecturii Agent Framework

Microsoft Agent Framework urmează o arhitectură stratificată:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Un `FoundryChatClient` se conectează la o implementare Azure OpenAI. Se ocupă de autentificare, formatarea cererii și analiza răspunsului.
2. **Agent** – Creat din client prin `provider.create_agent()`, agentul combină accesul la model cu instrucțiuni (prompt de sistem) și unelte.
3. **Unelte** – Funcții Python decorate cu `@tool` pe care agentul le poate invoca pentru a executa acțiuni sau a prelua date.
4. **Sesiune** – Un obiect `AgentSession` (creat prin `agent.create_session()`) care stochează istoricul conversației, permițând dialoguri multi-turn, unde agentul își amintește contextul anterior.

Să construim fiecare strat pas cu pas.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Adăugarea instrumentelor cu decoratorul @tool

Instrumentele permit agenților să efectueze acțiuni dincolo de generarea de text. Decoratorul `@tool` transformă o funcție obișnuită Python într-un ceva ce agentul poate apela.

Puncte cheie:
- Folosiți `Annotated[type, "descriere"]` pentru ca modelul să înțeleagă fiecare parametru.
- Docstring-ul devine descrierea instrumentului pe care modelul o vede.
- `approval_mode="never_require"` înseamnă că instrumentul rulează automat fără confirmarea utilizatorului.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Crearea unui Agent cu Unelte

Acum combinăm clientul, instrucțiunile și uneltele într-un agent. `instructions` acționează ca promptul sistemului — ele definesc persona și comportamentul agentului.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversații multi-tur cu sesiuni

O `AgentSession` (creată prin `agent.create_session()`) păstrează evidența tuturor mesajelor dintr-o conversație. Prin transmiterea aceleiași sesiuni la fiecare apel `agent.run()`, agentul are acces la întreg istoricul conversației și se poate referi la mesajele anterioare.

Transmitem `tools=[check_destination_availability]` astfel încât agentul să poată apela verificatorul nostru de disponibilitate în fiecare tur.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Rezumat

În această lecție ai explorat cele patru piloni ai Framework-ului Microsoft Agent:

| Concept | Ce ai învățat |
|---------|------------------|
| **Client** | `FoundryChatClient` se conectează la Azure OpenAI cu autentificare pe bază de credentiale |
| **Agent** | `provider.create_agent()` grupează o conexiune la model cu instrucțiuni și un nume |
| **Unelte** | Decoratorul `@tool` expune funcții Python pentru ca agentul să le poată apela |
| **Sesiune** | `agent.create_session()` păstrează istoricul conversației pe mai multe schimburi |

Aceste componente construiesc împreună agenți care pot purta conversații naturale, pot apela funcții externe și pot menține contextul — fundamentul pentru tipare agentice mai avansate în lecțiile viitoare.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
